In [7]:
import os
import numpy as np
from PIL import Image

In [8]:
print(os.listdir())

['.ipynb_checkpoints', 'Manas Task 4.0.ipynb', 'Test', 'Train']


In [9]:
def load_dataset(folder_path):
    X = []
    y = []
    
    class_names = sorted(os.listdir(folder_path))
    print("Classes:", class_names)
    
    label_map = {name: idx for idx, name in enumerate(class_names)}
    
    for class_name in class_names:
        class_path = os.path.join(folder_path, class_name)
        
        print("Checking folder:", class_path)
        
        if not os.path.isdir(class_path):
            print("Not a directory, skipping")
            continue
        
        files = os.listdir(class_path)
        print("Number of files:", len(files))
        
        # Show sample files
        for file in files[:5]:
            print("Sample file:", file)
        
        for file in files:
            # Skip non-image files
            if not file.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue
            
            img_path = os.path.join(class_path, file)
            
            try:
                img = Image.open(img_path).convert('L')  # grayscale
                img = img.resize((28, 28))               # ensure size
                img = np.array(img) / 255.0              # normalize
                img = img.flatten()                      # (784,)
                
                X.append(img)
                y.append(label_map[class_name])
            
            except Exception as e:
                print("Error loading:", img_path)
                print(e)
    
    return np.array(X), np.array(y), label_map

In [10]:
X_train, y_train, label_map = load_dataset("Train/Train")
X_test, y_test, _ = load_dataset("Test/Test")

Classes: ['Jade', 'James', 'Jane', 'Joel', 'Jovi']
Checking folder: Train/Train\Jade
Number of files: 800
Sample file: 0.png
Sample file: 1.png
Sample file: 10.png
Sample file: 100.png
Sample file: 101.png
Checking folder: Train/Train\James
Number of files: 800
Sample file: 0.png
Sample file: 1.png
Sample file: 10.png
Sample file: 100.png
Sample file: 101.png
Checking folder: Train/Train\Jane
Number of files: 800
Sample file: 0.png
Sample file: 1.png
Sample file: 10.png
Sample file: 100.png
Sample file: 101.png
Checking folder: Train/Train\Joel
Number of files: 800
Sample file: 0.png
Sample file: 1.png
Sample file: 10.png
Sample file: 100.png
Sample file: 101.png
Checking folder: Train/Train\Jovi
Number of files: 800
Sample file: 0.png
Sample file: 1.png
Sample file: 10.png
Sample file: 100.png
Sample file: 101.png
Classes: ['Jade', 'James', 'Jane', 'Joel', 'Jovi']
Checking folder: Test/Test\Jade
Number of files: 201
Sample file: -601.png
Sample file: -602.png
Sample file: -603.png
Sam

In [11]:
mean = np.mean(X_train, axis=0)

X_train = X_train - mean
X_test = X_test - mean

In [12]:
#verifying shape
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(4000, 784) (4000,)
(1000, 784) (1000,)


In [13]:
#shuffling the training data so that the model doesnt learn the order
perm = np.random.permutation(len(X_train))
X_train = X_train[perm]
y_train = y_train[perm]

In [14]:
#one hot label encoding
def one_hot(y, num_classes=5):
    return np.eye(num_classes)[y]

y_train_oh = one_hot(y_train)
y_test_oh = one_hot(y_test)

In [15]:
#activation function:relu (turns negatives to zero and keeps the positives)
def relu(Z):
    return np.maximum(0, Z)

In [16]:
#activation function: softmax (to convert output to probabilities)
def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=1, keepdims=True))
    return expZ / np.sum(expZ, axis=1, keepdims=True)

In [17]:
#weight initialization
W1 = np.random.randn(784, 128) * 0.01
b1 = np.zeros((1, 128))

W2 = np.random.randn(128, 64) * 0.01
b2 = np.zeros((1, 64))

W3 = np.random.randn(64, 5) * 0.01
b3 = np.zeros((1, 5))

In [18]:
#forward pass
def forward_pass(X, W1, b1, W2, b2, W3, b3):
    
    # Layer 1
    Z1 = np.dot(X, W1) + b1
    A1 = relu(Z1)
    
    # Layer 2
    Z2 = np.dot(A1, W2) + b2
    A2 = relu(Z2)
    
    # Output Layer
    Z3 = np.dot(A2, W3) + b3
    A3 = softmax(Z3)
    
    return Z1, A1, Z2, A2, Z3, A3

In [19]:
Z1, A1, Z2, A2, Z3, A3 = forward_pass(X_train, W1, b1, W2, b2, W3, b3)

In [20]:
print(A3.shape)
print(np.sum(A3[0]))

(4000, 5)
1.0


In [27]:
def compute_loss(y_true, y_pred):
    m = y_true.shape[0]
    loss = -np.sum(y_true * np.log(y_pred + 1e-8)) / m
    return float(loss)

In [22]:
def relu_derivative(Z):
    return (Z > 0).astype(float)


def backward_pass(X, y, Z1, A1, Z2, A2, Z3, A3, W2, W3):
    m = X.shape[0]

    # for Output layer
    dZ3 = A3 - y                      
    dW3 = np.dot(A2.T, dZ3) / m       
    db3 = np.sum(dZ3, axis=0, keepdims=True) / m

    # Layer 2
    dA2 = np.dot(dZ3, W3.T)           # (m, 64)
    dZ2 = dA2 * relu_derivative(Z2)
    dW2 = np.dot(A1.T, dZ2) / m       # (128, 64)
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    # Layer 1
    dA1 = np.dot(dZ2, W2.T)           # (m, 128)
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = np.dot(X.T, dZ1) / m        # (784, 128)
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    return dW1, db1, dW2, db2, dW3, db3

In [23]:
#updating parameters
def update_params(W1, b1, W2, b2, W3, b3,
                  dW1, db1, dW2, db2, dW3, db3,
                  lr):

    W1 -= lr * dW1
    b1 -= lr * db1

    W2 -= lr * dW2
    b2 -= lr * db2

    W3 -= lr * dW3
    b3 -= lr * db3

    return W1, b1, W2, b2, W3, b3

In [34]:
epochs = 100
lr = 0.6

for epoch in range(epochs):

    #Forward pass
    Z1, A1, Z2, A2, Z3, A3 = forward_pass(X_train, W1, b1, W2, b2, W3, b3)

    #Loss
    loss = compute_loss(y_train_oh, A3)

    #Backprop
    dW1, db1, dW2, db2, dW3, db3 = backward_pass(
        X_train, y_train_oh, Z1, A1, Z2, A2, Z3, A3, W2, W3
    )

    #Update
    W1, b1, W2, b2, W3, b3 = update_params(
        W1, b1, W2, b2, W3, b3,
        dW1, db1, dW2, db2, dW3, db3,
        lr
    )

    #Printing progress
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

Epoch 0, Loss: 0.6980
Epoch 10, Loss: 0.2513
Epoch 20, Loss: 0.2223
Epoch 30, Loss: 0.1886
Epoch 40, Loss: 0.1624
Epoch 50, Loss: 0.1416
Epoch 60, Loss: 0.1260
Epoch 70, Loss: 0.1119
Epoch 80, Loss: 0.1010
Epoch 90, Loss: 0.0920


In [36]:
def compute_accuracy(y_true, y_pred):
    y_true_labels = np.argmax(y_true, axis=1)
    y_pred_labels = np.argmax(y_pred, axis=1)
    return np.mean(y_true_labels == y_pred_labels)

In [37]:
train_acc = compute_accuracy(y_train_oh, A3)
print("Train Accuracy:", train_acc)

Train Accuracy: 0.97275


In [38]:
_, _, _, _, _, A3_test = forward_pass(X_test, W1, b1, W2, b2, W3, b3)

test_acc = compute_accuracy(y_test_oh, A3_test)
print("Test Accuracy:", test_acc)

Test Accuracy: 0.959
